<a href="https://colab.research.google.com/github/Thilac01/Statistical-Learning-e22395/blob/main/Statistical_Learning_Assignement7c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Bayesian Estimation of a User Ability Parameter from Item Responses

# Adaptive Testing with the 2PL IRT Model
Sequential Bayesian ability tracking: update posterior over Θ after each response,
track convergence over 20 items.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def p_2pl(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

## Task 1: 2PL Item Characteristic Curves
Left: two discrimination values (a). Right: fixed a=1.5, three difficulty values (b).

In [ ]:
theta_grid = np.linspace(-4, 4, 400)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Effect of Discrimination (a)", "Effect of Difficulty (b), a=1.5"))

for a_val in [0.5, 2.0]:
    fig.add_trace(go.Scatter(x=theta_grid, y=p_2pl(theta_grid, a_val, 0),
                              mode="lines", name=f"a={a_val}, b=0"), row=1, col=1)

for b_val in [-1, 0, 1]:
    fig.add_trace(go.Scatter(x=theta_grid, y=p_2pl(theta_grid, 1.5, b_val),
                              mode="lines", name=f"a=1.5, b={b_val}"), row=1, col=2)

fig.update_xaxes(title_text="θ (ability)")
fig.update_yaxes(title_text="P(Y=1 | θ)")
fig.update_layout(title="2PL Item Characteristic Curves", height=450, width=950)
fig.show()

## Task 2: Likelihood Contribution
Single response:
L(y_k | θ) = p_k(θ)^y_k * (1 - p_k(θ))^(1 - y_k)

Joint likelihood for running history y^(k):
L(y^(k) | θ) = Π_{i=1}^{k} p_i(θ)^y_i * (1 - p_i(θ))^(1 - y_i)

## Task 3: Recursive Posterior Update
f(θ | y^(k)) ∝ f(θ | y^(k-1)) * L(y_k | θ)

The old posterior becomes the new prior — no need to reprocess full history.

## Task 4: Dynamic Shifting
y_k = 1 contributes factor p_k(θ), increasing in θ → pulls posterior mode right.
Large b_k means the "success region" itself sits at high θ, so a correct answer on
a hard item gives strong evidence for high ability → bigger rightward shift than
an easy item would produce.

## Task 5: Discrimination and Sharpness
a_k controls the slope of p_k(θ) near b_k:
- Large a_k → likelihood behaves like a step function → posterior becomes much
  sharper (lower variance) → fast convergence.
- Small a_k → likelihood nearly flat/constant → posterior barely changes from
  the prior → little information gained.

## Task 6: Numerical Grid Implementation
1. Fix a grid θ_1...θ_m over [-6, 6].
2. Initialize f^(0)(θ_j) = N(0,1) density at each grid point.
3. For each new item (a_k, b_k, y_k):
   - Compute likelihood L_j = p_k(θ_j)^y_k * (1-p_k(θ_j))^(1-y_k)
   - Unnormalized posterior: f_tilde_j = f^(k-1)(θ_j) * L_j
   - Normalize: f^(k)(θ_j) = f_tilde_j / (Σ_j f_tilde_j * Δθ)   [Riemann sum integration]
4. Posterior Mean = Σ θ_j f^(k)(θ_j) Δθ
   MAP = argmax_j f^(k)(θ_j)

In [ ]:
theta_axis = np.linspace(-6, 6, 1201)
dtheta = theta_axis[1] - theta_axis[0]

def normal_pdf(x, mu=0, sigma=1):
    return (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((x-mu)/sigma)**2)

def update_posterior(prior_density, a_k, b_k, y_k, grid=theta_axis, dx=dtheta):
    p_k = p_2pl(grid, a_k, b_k)
    likelihood = np.where(y_k == 1, p_k, 1 - p_k)
    unnorm_post = prior_density * likelihood
    norm_const = np.sum(unnorm_post) * dx
    return unnorm_post / norm_const

## Task 7: Simulating 20 Items (θ_true = 0.75)
Random b_k ~ N(0,1), a_k ~ Uniform(0.5, 2.0). Simulate y_k, track running
Posterior Mean and MAP estimates.

In [ ]:
np.random.seed(42)

theta_true = 0.75
n_items = 20

b_items = np.random.normal(0, 1, n_items)
a_items = np.random.uniform(0.5, 2.0, n_items)

posterior = normal_pdf(theta_axis, mu=0, sigma=1)
posterior_means = [np.sum(theta_axis * posterior) * dtheta]
map_estimates   = [theta_axis[np.argmax(posterior)]]
responses = []

for k in range(n_items):
    a_k, b_k = a_items[k], b_items[k]
    p_true = p_2pl(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0
    responses.append(y_k)

    posterior = update_posterior(posterior, a_k, b_k, y_k)
    posterior_means.append(np.sum(theta_axis * posterior) * dtheta)
    map_estimates.append(theta_axis[np.argmax(posterior)])

print("Responses:", responses)
print("Final Posterior Mean:", round(posterior_means[-1], 4))
print("Final MAP estimate:  ", round(map_estimates[-1], 4))

In [ ]:
steps = list(range(0, n_items + 1))

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=steps, y=posterior_means, mode="lines+markers",
                           name="Posterior Mean (Bayes)", line=dict(color="royalblue")))
fig2.add_trace(go.Scatter(x=steps, y=map_estimates, mode="lines+markers",
                           name="MAP Estimate", line=dict(color="orange")))
fig2.add_hline(y=theta_true, line_dash="dash", line_color="green",
               annotation_text="θ_true = 0.75", annotation_position="bottom right")

fig2.update_layout(title="Convergence of Ability Estimators over 20 Items",
                    xaxis_title="Item Step (k)", yaxis_title="Estimated Ability (θ)",
                    height=500, width=850)
fig2.show()

## Analysis
As k increases, both estimators generally move closer to θ_true = 0.75, though not
strictly monotonically — an unexpected response can cause a temporary jump. Early
on, the posterior is close to the wide N(0,1) prior, so each response has a large
effect. As more items accumulate, posterior variance shrinks (faster with
high-discrimination items), each new update moves the peak less, and the mean and
MAP converge toward each other and toward θ_true. This shows the platform's
confidence (precision) grows steadily even if the point estimate wobbles slightly
step to step.

# Bayesian CTR Estimation with Beta-Binomial Conjugacy
Sequential updating of click-through rate belief, one impression at a time,
using the Beta distribution as a conjugate prior for a Bernoulli likelihood.

----
----

In [ ]:
import numpy as np
from scipy.stats import beta as beta_dist
import plotly.graph_objects as go

## Task 1: Beta Distribution Shapes
Three parameter pairs: uninformative (1,1), right-skewed (2,8), left-skewed (8,2).

In [ ]:
theta_grid = np.linspace(0, 1, 400)

params = [(1, 1, "Uninformative: α=1, β=1"),
          (2, 8, "Right-skewed: α=2, β=8"),
          (8, 2, "Left-skewed: α=8, β=2")]

fig = go.Figure()
for a, b, label in params:
    pdf = beta_dist.pdf(theta_grid, a, b)
    fig.add_trace(go.Scatter(x=theta_grid, y=pdf, mode="lines", name=label))

fig.update_layout(title="Beta(α, β) Density for Different Parameter Pairs",
                   xaxis_title="θ", yaxis_title="Density f(θ)",
                   height=500, width=850)
fig.show()

## Task 2: Likelihood Contribution
Single response likelihood (Bernoulli):
L(y_k | θ) = θ^y_k * (1 - θ)^(1 - y_k)

Joint likelihood for running history y^(k):
L(y^(k) | θ) = Π_{i=1}^{k} θ^y_i * (1 - θ)^(1 - y_i) = θ^S_k * (1 - θ)^(k - S_k)

where S_k = Σ y_i is the running number of clicks up to step k.

## Task 3: Closed-Form Update (Beta-Binomial Conjugacy)
By Bayes' Theorem:
f(θ | y^(k)) ∝ f(θ | y^(k-1)) * L(y_k | θ)
            ∝ θ^(α_{k-1}-1) (1-θ)^(β_{k-1}-1) * θ^{y_k} (1-θ)^{1-y_k}
            = θ^(α_{k-1}+y_k-1) (1-θ)^(β_{k-1}+1-y_k-1)

This has exactly the Beta kernel form, so the posterior stays in the Beta family
(conjugacy) with updated parameters:

  α_k = α_{k-1} + y_k
  β_k = β_{k-1} + (1 - y_k)

i.e. add 1 to α on a click, add 1 to β on a non-click — pure integer bookkeeping.

Posterior Mean at step k:
  E[Θ | y^(k)] = α_k / (α_k + β_k)

## Task 4: Dynamic Shifting & Contrast with Non-Conjugate Models
A click (y_k=1) increments α_k, shifting mass toward higher θ and moving the peak
right. A non-click (y_k=0) increments β_k, shifting mass toward lower θ and moving
the peak left. The magnitude of each shift naturally shrinks as k grows, since
α_{k-1}+β_{k-1} (the "pseudo-count" of prior evidence) grows with each step,
diluting the relative effect of any single new observation.

Contrast with 2PL IRT: there, the likelihood factor depends on item-specific
parameters (a_k, b_k) and does not combine algebraically with a Normal prior to
produce another Normal — the posterior has no closed form, so numerical grid
integration and renormalization are required at every step. Here, Beta-Binomial
conjugacy means the *entire* posterior update reduces to two integer additions,
with no numerical integration needed at all.

## Task 5: Running Point Estimators (Closed Form)
Posterior Mean:
  θ_Bayes^(k) = α_k / (α_k + β_k)

MAP (mode of Beta, valid for α_k, β_k > 1):
  θ_MAP^(k) = (α_k - 1) / (α_k + β_k - 2)

(If α_k ≤ 1 or β_k ≤ 1, the mode is at a boundary, 0 or 1, instead.)

## Task 6: Simulating 100 Impressions (θ_true = 0.35)
Initialize α_0 = β_0 = 1. Simulate clicks, update α_k, β_k analytically,
track Posterior Mean and MAP at every step.

In [ ]:
np.random.seed(42)

theta_true = 0.35
n_impressions = 100

alpha_0, beta_0 = 1, 1
alpha_k, beta_k = alpha_0, beta_0

posterior_means = [alpha_k / (alpha_k + beta_k)]
map_estimates   = [ (alpha_k - 1) / (alpha_k + beta_k - 2) if (alpha_k > 1 and beta_k > 1) else posterior_means[0] ]
responses = []

for k in range(n_impressions):
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    responses.append(y_k)

    alpha_k += y_k
    beta_k  += (1 - y_k)

    posterior_means.append(alpha_k / (alpha_k + beta_k))

    if alpha_k > 1 and beta_k > 1:
        map_k = (alpha_k - 1) / (alpha_k + beta_k - 2)
    else:
        map_k = 0.0 if alpha_k < beta_k else 1.0
    map_estimates.append(map_k)

print("Total clicks:", sum(responses), "/", n_impressions)
print("Final alpha_k, beta_k:", alpha_k, beta_k)
print("Final Posterior Mean:", round(posterior_means[-1], 4))
print("Final MAP estimate:  ", round(map_estimates[-1], 4))

In [ ]:
steps = list(range(0, n_impressions + 1))

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=steps, y=posterior_means, mode="lines",
                           name="Posterior Mean (Bayes)", line=dict(color="royalblue")))
fig2.add_trace(go.Scatter(x=steps, y=map_estimates, mode="lines",
                           name="MAP Estimate", line=dict(color="orange")))
fig2.add_hline(y=theta_true, line_dash="dash", line_color="green",
               annotation_text="θ_true = 0.35", annotation_position="bottom right")

fig2.update_layout(title="Convergence of CTR Estimators over 100 Impressions",
                    xaxis_title="Impression Step (k)", yaxis_title="Estimated CTR (θ)",
                    height=500, width=850)
fig2.show()

## Analysis
As k approaches 100, both the Posterior Mean and MAP estimate converge toward
θ_true = 0.35, with the gap between them and the true value shrinking steadily.
Early on, with only a handful of impressions, the Beta(1,1) uniform prior still
carries substantial relative weight, so each new click or non-click causes large
swings in the estimate. As k grows, α_k + β_k (the effective total pseudo-count of
evidence) increases, so each new observation contributes a proportionally smaller
adjustment — the posterior variance α_k β_k / [(α_k+β_k)^2 (α_k+β_k+1)] shrinks
roughly at rate 1/k.

This illustrates a core property of conjugate Bayesian updating: the influence of
the initial prior choice (α_0, β_0) fades asymptotically as data accumulates — by
k=100, the observed click history dominates the posterior almost entirely,
regardless of whether the prior started uninformative or mildly biased. In
practice, this means the platform's estimate becomes increasingly trustworthy and
stable the longer the ad runs, even though early estimates should be treated with
caution due to their higher variance.